# Series and DataFrames

## Learning objectives

By the end of this notebook you will be able to:

- create Series and DataFrames from lists, dictionaries and arrays
- understand the index and how operations align on it
- access columns, and add and remove them
- inspect data types

## Introduction

Before loading real files it helps to build small tables by hand, so you can see exactly how pandas
stores rows, columns and the index.

In [1]:
import pandas as pd
import numpy as np
RAW = "../datasets/raw/"
PROCESSED = "../datasets/processed/"

## Series

In [2]:
glucose = pd.Series([5.4, 7.8, 6.1, 9.2])
print(glucose)

0    5.4
1    7.8
2    6.1
3    9.2
dtype: float64


The left column is the **index**. By default it is 0, 1, 2, ... but it can be any labels:

In [3]:
glucose = pd.Series([5.4, 7.8, 6.1, 9.2], index=["P01", "P02", "P03", "P04"], name="glucose")
print(glucose)
print(glucose["P03"], glucose.iloc[0])

P01    5.4
P02    7.8
P03    6.1
P04    9.2
Name: glucose, dtype: float64
6.1 5.4


A Series has NumPy-style vectorised maths and statistics methods:

In [4]:
print((glucose * 18).round(0))
print(glucose.mean(), glucose.max(), glucose.idxmax())
print(glucose[glucose > 7])

P01     97.0
P02    140.0
P03    110.0
P04    166.0
Name: glucose, dtype: float64
7.124999999999999 9.2 P04
P02    7.8
P04    9.2
Name: glucose, dtype: float64


### Alignment on the index

Operations between Series match labels, not positions. Labels missing from either side give NaN.

In [5]:
before = pd.Series({"P01": 142, "P02": 128, "P03": 155})
after = pd.Series({"P03": 141, "P01": 131, "P04": 120})
print(after - before)

P01   -11.0
P02     NaN
P03   -14.0
P04     NaN
dtype: float64


P01 and P03 are matched correctly even though they are in a different order; P02 and P04 have no
partner, so their result is missing. This prevents silent mismatches, but means you should check for
unexpected NaN after combining data.

## DataFrames

### From a dictionary of columns

In [6]:
df = pd.DataFrame({
    "patient_id": ["P01", "P02", "P03", "P04"],
    "sex": ["F", "M", "F", "M"],
    "age": [34, 51, 47, 62],
    "glucose": [5.4, 7.8, 6.1, 9.2],
})
df

,patient_id,sex,age,glucose
0,P01,F,34,5.4
1,P02,M,51,7.8
2,P03,F,47,6.1
3,P04,M,62,9.2


### From a list of dictionaries (one per row)

In [7]:
rows = [
    {"patient_id": "P05", "sex": "F", "age": 29, "glucose": 5.0},
    {"patient_id": "P06", "sex": "M", "age": 44},
]
pd.DataFrame(rows)

,patient_id,sex,age,glucose
0,P05,F,29,5.0
1,P06,M,44,NaN


### From a NumPy array

In [8]:
rng = np.random.default_rng(0)
pd.DataFrame(rng.normal(10, 2, (3, 4)).round(1), columns=["run1", "run2", "run3", "run4"],
             index=["sample_a", "sample_b", "sample_c"])

,run1,run2,run3,run4
sample_a,10.3,9.7,11.3,10.2
sample_b,8.9,10.7,12.6,11.9
sample_c,8.6,7.5,8.8,10.1


## Anatomy of a DataFrame

In [9]:
print(df.shape)
print(df.columns)
print(df.index)
print(df.dtypes)

(4, 4)
Index(['patient_id', 'sex', 'age', 'glucose'], dtype='str')
RangeIndex(start=0, stop=4, step=1)
patient_id        str
sex               str
age             int64
glucose       float64
dtype: object


pandas uses `int64` and `float64` for numbers, `str` for text (pandas 3) or `object` (older versions),
`bool`, `datetime64` for dates and `category` for categories.

## Columns

`df["name"]` returns a Series; `df[["a", "b"]]` (a list) returns a DataFrame.

In [10]:
print(df["age"])
df[["patient_id", "glucose"]]

0    34
1    51
2    47
3    62
Name: age, dtype: int64


,patient_id,glucose
0,P01,5.4
1,P02,7.8
2,P03,6.1
3,P04,9.2


Adding and changing columns:

In [11]:
df["glucose_mg_dl"] = (df["glucose"] * 18).round(0)
df["high"] = df["glucose"] >= 7
df["site"] = "Site A"                       # one value for every row
df

,patient_id,sex,age,glucose,glucose_mg_dl,high,site
0,P01,F,34,5.4,97.0,False,Site A
1,P02,M,51,7.8,140.0,True,Site A
2,P03,F,47,6.1,110.0,False,Site A
3,P04,M,62,9.2,166.0,True,Site A


Removing columns and renaming them:

In [12]:
df = df.drop(columns=["site"])
df = df.rename(columns={"glucose": "glucose_mmol"})
df.columns.tolist()

['patient_id', 'sex', 'age', 'glucose_mmol', 'glucose_mg_dl', 'high']

## Setting the index

A column of unique IDs often makes a good index.

In [13]:
indexed = df.set_index("patient_id")
print(indexed.loc["P03"])
indexed.reset_index().head(2)

sex                  F
age                 47
glucose_mmol       6.1
glucose_mg_dl    110.0
high             False
Name: P03, dtype: object


,patient_id,sex,age,glucose_mmol,glucose_mg_dl,high
0,P01,F,34,5.4,97.0,False
1,P02,M,51,7.8,140.0,True


## Rows

`len(df)` counts rows; `df.iloc[0]` is the first row; `df.loc[label]` selects by index label
(details in notebook 06).

In [14]:
print(len(df))
print(df.iloc[0])

4
patient_id         P01
sex                  F
age                 34
glucose_mmol       5.4
glucose_mg_dl     97.0
high             False
Name: 0, dtype: object


## Practise

Exercises for this folder are in `exercises/Pandas_Exercises.ipynb`.

## Further reading

- [pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html)
- [Intro to data structures](https://pandas.pydata.org/docs/user_guide/dsintro.html)